# Man-in-the-Middle — Selective Hijack in a Randomized Fleet

Three drones start at **randomized ground positions** and fly **randomized
trajectories**, all monitored by a **single GCS process** (they share one
`SimGCS`). An attacker has man-in-the-middled **only the middle drone (sysid
2)**: once it reaches a chosen waypoint (`MISSION_CURRENT.seq >= trigger_seq`)
the MITM injects `SET_MODE(GUIDED)` + `DO_REPOSITION` toward attacker-chosen
coordinates, spoofed to look like the GCS (sysid 255).

**Keeping the fleet realistic and safe** (see the layout cell for the knobs):

- **Close together** — every home is sampled inside a disk of `FLEET_RADIUS`
  around the origin, so all drones stay well inside the GCS transmission range.
- **No collision on the ground** — homes are rejection-sampled to stay at least
  `MIN_SEPARATION` apart. All homes have `z=0`, so every drone starts landed.
- **No collision in the air** — each drone flies in its own cruise-altitude
  layer (`BASE_ALT + i·ALT_STEP`), so random horizontal paths can cross without
  the drones ever sharing a point in 3D.

The other two drones have **no MITM** — their telemetry and commands reach the
GCS directly. The point of this scenario is that `simulator.mitm` is keyed by
sysid, so one vehicle can be compromised while its fleet-mates on the same GCS
are untouched. Change `SEED` in the layout cell for a fresh random fleet.

In [ ]:
import math
import random

from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, PARAMS_PATH, Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin, fleet layout, and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

speed = 5.0  # m/s
model = Model.IRIS

sysids = [1, 2, 3]
colors = [Color.GREEN, Color.BLUE, Color.RED]

# Remove SEED for experiments
SEED = 42
rng = random.Random(SEED)


FLEET_RADIUS = 20.0  # m  — how far a home may sit from the origin
MIN_SEPARATION = 6.0  # m  — closest two homes may sit
BASE_ALT = 8.0  # m  — lowest drone's cruise altitude
ALT_STEP = 4.0  # m  — vertical spacing between drones
HIJACK_RADIUS = 40.0  # m  — attacker's redirect target sampled within this disk


def sample_disk(radius: float) -> tuple[float, float]:
    # sqrt gives a uniform (not center-biased) distribution over the disk
    r = radius * math.sqrt(rng.random())
    theta = rng.uniform(0.0, 2 * math.pi)
    return r * math.cos(theta), r * math.sin(theta)


def sample_home(placed: list[ENUPose]) -> ENUPose:
    while True:
        east, north = sample_disk(FLEET_RADIUS)
        if all(math.hypot(east - h.x, north - h.y) >= MIN_SEPARATION for h in placed):
            heading = rng.uniform(0.0, 360.0)
            return ENUPose(east, north, 0.0, heading)  # z=0 → starts on the ground


def random_mission(alt: float) -> list[ENU]:
    wps = [ENU(x=0, y=0, z=0)]  # home / takeoff point on the ground
    east, north = 0.0, 0.0
    for _ in range(rng.randint(2, 3)):
        az = rng.uniform(0.0, 2 * math.pi)
        leg = rng.uniform(15.0, 30.0)  # metres per leg
        east += leg * math.cos(az)
        north += leg * math.sin(az)
        wps.append(ENU(x=east, y=north, z=alt))  # whole path stays at `alt`
    return wps


# One home per drone (ground-separated), one altitude layer per drone.
homes: list[ENUPose] = []
for _ in sysids:
    homes.append(sample_home(homes))
cruise_alts = {sysid: BASE_ALT + i * ALT_STEP for i, sysid in enumerate(sysids)}
missions = {sysid: random_mission(cruise_alts[sysid]) for sysid in sysids}

for sysid, home in zip(sysids, homes, strict=True):
    print(
        f"Drone {sysid}: home=({home.x:+.1f}, {home.y:+.1f}) m, "
        f"heading={home.heading:.0f}°, alt={cruise_alts[sysid]:.0f} m, "
        f"{len(missions[sysid]) - 1} legs"
    )

hijacked_sysid = 1
hijack_alt = cruise_alts[hijacked_sysid]
# Random redirect target within HIJACK_RADIUS of the origin, at the hijacked
# drone's own altitude layer (so the hijack can't cause a collision either).
hijack_east, hijack_north = sample_disk(HIJACK_RADIUS)
hijack_target = gra_origin.unpose().to_abs(
    ENU(x=hijack_east, y=hijack_north, z=hijack_alt)
)
print(
    f"Hijacking sysid {hijacked_sysid} → offset "
    f"({hijack_east:+.1f}, {hijack_north:+.1f}) m "
    f"[{math.hypot(hijack_east, hijack_north):.1f} m from origin], "
    f"lat={hijack_target.lat:.7f}, lon={hijack_target.lon:.7f}, alt={hijack_alt:.0f} m"
)

## Vehicles (all on one shared GCS)

In [ ]:
# A single shared SimGCS instance puts all three vehicles under one GCS process.
gcs = SimGCS(name=f"FLEET_{''.join(color.emoji for color in colors)}")

mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

vehs: list[SimVehicle] = []
for sysid, color, home in zip(sysids, colors, homes, strict=True):
    mission_wps = missions[sysid]
    mission_path = str(mission_folder / f"mitm_fleet_rand_{sysid}.waypoints")
    plan = AutoPlan.from_relative_path(
        name="random_mission",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=home,
        relative_path=mission_wps,
        mission_path=mission_path,
        navigation_speed=speed,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs=gcs,
        plan=plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=home,
        relative_path=mission_wps,
        model=model,
    )
    vehs.append(veh)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + selective MITM hijack

In [ ]:
simulator = Simulator(visualizer=gaz, verbose=1)
for veh in vehs:
    simulator.add_vehicle(veh, parm=str(PARAMS_PATH / "vehicle.parm"))

simulator.mitm[hijacked_sysid] = {
    "strategy": "hijack",
    "params": {
        "trigger_seq": 3,
        "target_lat": hijack_target.lat,
        "target_lon": hijack_target.lon,
        "target_alt": hijack_alt,
    },
}

simulator.preview()

In [ ]:
simulator.launch()
orac = Oracle(
    simulator.gra_origin,
    simulator.vehicles,
    simulator.gcs,
    port_offset=simulator.orc_port_offset,
)
orac.run()

After the run: `python -m tools.plot_gcs_telemetry` shows the fleet's
trajectories (longitude vs latitude) as reconstructed from GCS telemetry.